# Streams

Welcome to the documentation the *Streams* part of *Kafi Streams*.

Streams offers easy-as-cake complex stateful stream processing based on pydbsp in the spirit of Kafka Streams.

## Topology

In Streams, you first create a *processing topology* or just *topology* with an arbitrary number of *sources* (=inputs) and an arbitrary number of *sinks* (=outputs).

A topology is written using a fluent API inspired by Kafka Streams (which, in turn, has been inspired by Java 8+ Streams).

The underlying class is called *TopologyNode* - which is basically just a thin wrapper around pydbsp.


### An Example Topology

Let's proceed to an example topology:

In [32]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

click_source_str = "clicks"
customer_source_str = "customers"
sink_str = "join_1"
#
click_tn = (
    Tn.source(click_source_str)
    .map(lambda x: {"customer_id": x["value"]["customer_id"], "ip": x["value"]["ip"]})
    .distinct()
)
#
customer_tn = (
    Tn.source(customer_source_str)
    .map(lambda x: {"id": x["value"]["id"], "name": x["value"]["name"]})
    .distinct()
)
#
join_1_tn = (
    click_tn
    .join_equi(
        customer_tn,
        lambda l: l["customer_id"],
        lambda r: r["id"],
        lambda l, r: {"value": {
            "customer_id": l["customer_id"],
            "ip": l["ip"],
            "name": r["name"]}})
    .sink(sink_str)
)
#
built_tn = Tn.build(join_1_tn)


So what happens here? Let's have a look at the topology in visual form:


In [33]:
print(built_tn.mermaid())

```mermaid
graph TD
b903e26b-a9e1-4d76-a0d0-af9427019696[map_op] --> 3dea6074-0fce-4007-8efa-0c499ef66563[distinct_op]
8f7ef99c-0e0a-4b1d-ad8d-86957485e526[join_equi_op] --> 13f25140-11c7-4beb-9bca-17a30aa9a155[sink_join_1]
64bae356-f252-40f7-a054-5f0b7abc68fe[source_clicks] --> 8894044d-22ce-4975-9cbd-1bb868660f96[map_op]
8894044d-22ce-4975-9cbd-1bb868660f96[map_op] --> bb683655-62c4-4950-8c44-9742ec74c431[distinct_op]
d775e1ce-ae9e-4640-892f-7faf148600a2[source_customers] --> b903e26b-a9e1-4d76-a0d0-af9427019696[map_op]
3dea6074-0fce-4007-8efa-0c499ef66563[distinct_op] --> 8f7ef99c-0e0a-4b1d-ad8d-86957485e526[join_equi_op]
bb683655-62c4-4950-8c44-9742ec74c431[distinct_op] --> 8f7ef99c-0e0a-4b1d-ad8d-86957485e526[join_equi_op]
```


```mermaid
graph TD
b903e26b-a9e1-4d76-a0d0-af9427019696[map_op] --> 3dea6074-0fce-4007-8efa-0c499ef66563[distinct_op]
8f7ef99c-0e0a-4b1d-ad8d-86957485e526[join_equi_op] --> 13f25140-11c7-4beb-9bca-17a30aa9a155[sink_join_1]
64bae356-f252-40f7-a054-5f0b7abc68fe[source_clicks] --> 8894044d-22ce-4975-9cbd-1bb868660f96[map_op]
8894044d-22ce-4975-9cbd-1bb868660f96[map_op] --> bb683655-62c4-4950-8c44-9742ec74c431[distinct_op]
d775e1ce-ae9e-4640-892f-7faf148600a2[source_customers] --> b903e26b-a9e1-4d76-a0d0-af9427019696[map_op]
3dea6074-0fce-4007-8efa-0c499ef66563[distinct_op] --> 8f7ef99c-0e0a-4b1d-ad8d-86957485e526[join_equi_op]
bb683655-62c4-4950-8c44-9742ec74c431[distinct_op] --> 8f7ef99c-0e0a-4b1d-ad8d-86957485e526[join_equi_op]
```

We have two sources...
* `clicks`
* `customers`

...and one sink:
* `join_1`

We assume that `clicks` is a source of Kafka messages like this...

In [ ]:
click_message_dict = {
    "key": None,
    "value": {"customer_id": "4711",
              "ip": "204.149.41.63"},
    "partition": 2,
    "offset": 23,
    "timestamp": 1609457201000,
    "headers": None
}

...and `customers` is a source of Kafka messages like this:

In [ ]:
customer_message_dict = {
    "key": "4711",
    "value": {"id": "4711",
              "name": "Sallyann Jupp"},
    "partition": 0,
    "offset": 67,
    "timestamp": 1609457001000,
    "headers": None
}

The aim of the topology is to join `clicks` with `customers` to associate the clickstream IPs with the respective customer. An example output message looks like this:

In [ ]:
{
    "value": {"customer_id": "4711",
              "ip": "204.149.41.63",
              "name": "Sallyann Jupp"}
}


#### How Does It Work?

How does the topology achieve its aim? It first selects the `customer_id` and `ip` fields from the `clicks` (`map`) and makes sure that the output of the selection contains no duplicates (`distinct`):

```python
click_tn = (
    Tn.source(click_source_str)
    .map(lambda x: {"customer_id": x["value"]["customer_id"], "ip": x["value"]["ip"]})
    .distinct()
)
```

Similarly, it selects `id` and `name` from the `customers` (`map` + `distinct`):
```python
customer_tn = (
    Tn.source(customer_source_str)
    .map(lambda x: {"id": x["value"]["id"], "name": x["value"]["name"]})
    .distinct()
)
```

Next, the topology does an equi join (`join_equi`) on `customer_id` from the `clicks` and `id` from the `customers`:
```python
join_1_tn = (
    click_tn
    .join_equi(
        customer_tn,
        lambda l: l["customer_id"],
        lambda r: r["id"],
        lambda l, r: {"value": {
            "customer_id": l["customer_id"],
            "ip": l["ip"],
            "name": r["name"]}})
    .sink(sink_str)
)
```
> You can use *any* field from the Kafka message for anything, including the equi join. We are not even restricted to `key` and `value`.

The final step "builds" the `TopologyNode` object by collecting its sinks:
```python
built_tn = Tn.build(join_1_tn)
```

That's it :-)


#### Let's Run It!

Now let's see if it works...

In [36]:
built_tn.push(click_source_str, [click_message_dict])
built_tn.push(customer_source_str, [customer_message_dict])
built_tn.latest()


{}

What happens if we push the same messages once again?

In [35]:
built_tn.push(click_source_str, [click_message_dict])
built_tn.push(customer_source_str, [customer_message_dict])
built_tn.latest()

{}

Nothing happens! Why?

Because nothing has changed! We did not add any further information, just the same information again. The `distinct()` operators did the magic for us.

In classical stream processing, we needed to implement the concept of *exactly-once semantics*.

> In Kafi Streams, you can forget about exactly-once semantics. It boils down to simply using the right operator in the right place.


#### But This Cannot Possibly Scale, Or Can It?

As you might have guessed, the `distinct()` operator (and also the `join_equi` operator, BTW) are stateful.

And yes - if we would confront this topology with an ever growing number of Kafka messages, we would soon run out of memory (and Kafi Streams would run into a crawl).

Let's see this happening.